1. Instalasi Library Terbaru

In [6]:
!pip install -q langchain langchain-chroma langchain-google-genai pypdf wandb python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.5 MB/s eta 0:00:00


2. Setup API Key & Weights & Biases (W&B)

In [7]:
import getpass
import os
import wandb

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Masukkan Gemini API Key: ")

if "WANDB_API_KEY" not in os.environ:
    os.environ["WANDB_API_KEY"] = getpass.getpass("Masukkan W&B API Key: ")

# Memulai logging ke dashboard Weights & Biases
wandb.init(project="tugas-ml-rag-gemini", name="evaluasi-pdf-magang")

Masukkan Gemini API Key: ··········
Masukkan W&B API Key: ··········


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: nisaagustin083 (nisaagustin083-stikomelrahma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
!pip install -q langchain-community chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 

3. Preprocessing Dokumen (Load, Chunking, & Vector DB)

In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
import os

# 1. Pastikan API Key terbaca dengan aman
api_key = os.environ.get("GOOGLE_API_KEY")



# 2. Load Dokumen PDF Magang
loader = PyPDFLoader("Panduan_Magang_El_Rahma_2026.pdf")
docs = loader.load()

# 3. Split Dokumen menjadi potongan kecil (Chunking)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=150
)
chunks = text_splitter.split_documents(docs)

# 4. Proses Embedding Menggunakan Model Standar yang Pasti Berhasil
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(f"Selesai! PDF dipecah menjadi {len(chunks)} chunks dan aman disimpan di Vector DB.")

Selesai! PDF dipecah menjadi 6 chunks dan aman disimpan di Vector DB.


4. Eksekusi Pengujian & Cetak Hasil (Baseline vs RAG)

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
    google_api_key=api_key
)

# Cari 3 potongan dokumen paling relevan (Top-K = 3)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Wadah tabel hasil untuk dikirim ke W&B
wb_table = wandb.Table(columns=["Pertanyaan", "Jawaban Tanpa RAG (Baseline)", "Jawaban Dengan RAG (Top-K=3)"])

def uji_sistem_rag(pertanyaan):
    # A. Uji Coba Tanpa RAG (Baseline)
    respon_baseline = llm.invoke(pertanyaan).content

    # B. Uji Coba Dengan RAG
    dokumen_relevan = retriever.invoke(pertanyaan)
    konteks = "\n\n".join([doc.page_content for doc in dokumen_relevan])

    prompt_lengkap = f"""Anda adalah asisten akademik yang akurat. Jawablah pertanyaan hanya berdasarkan KONTEKS yang diberikan.
    Jika jawabannya tidak tertera pada konteks, jawab 'Saya tidak tahu'. Jangan berasumsi atau mengarang jawaban.

    KONTEKS:
    {konteks}

    PERTANYAAN:
    {pertanyaan}
    """
    respon_rag = llm.invoke(prompt_lengkap).content

    # Simpan hasil uji coba ke tabel W&B
    wb_table.add_data(pertanyaan, respon_baseline, respon_rag)

    print(f"❓ PERTANYAAN:\n{pertanyaan}\n")
    print(f"❌ JAWABAN BASELINE (TANPA RAG):\n{respon_baseline}\n")
    print(f"✅ JAWABAN DENGAN RAG:\n{respon_rag}\n")
    print("="*60)

# Masukkan pertanyaan spesifik dari isi dokumen PDF-mu
pertanyaan_tugas = "Berapa target keuntungan bersih yang ditetapkan untuk Toko Niscare pada Kuartal Ke-4 tahun 2026 dan apa saja teknologi pendukungnya?"
uji_sistem_rag(pertanyaan_tugas)

# Kirim data tabel ke cloud W&B dan akhiri sesi logging
wandb.log({"tabel_evaluasi_rag": wb_table})
wandb.finish()

❓ PERTANYAAN:
Berapa target keuntungan bersih yang ditetapkan untuk Toko Niscare pada Kuartal Ke-4 tahun 2026 dan apa saja teknologi pendukungnya?

❌ JAWABAN BASELINE (TANPA RAG):
Sebagai sebuah entitas fiktif atau yang tidak memiliki informasi publik yang tersedia, **tidak ada target keuntungan bersih yang ditetapkan secara resmi untuk "Toko Niscare" pada Kuartal Ke-4 tahun 2026.**

Target keuntungan bersih adalah angka internal yang ditetapkan oleh manajemen perusahaan berdasarkan berbagai faktor seperti:
*   Kinerja historis
*   Proyeksi penjualan dan pertumbuhan pasar
*   Analisis biaya operasional
*   Strategi bisnis dan investasi yang direncanakan
*   Kondisi ekonomi makro

**Namun, jika kita berasumsi Toko Niscare adalah toko ritel modern yang ingin mencapai pertumbuhan dan efisiensi, berikut adalah contoh target hipotetis dan teknologi pendukung yang mungkin digunakan:**

---

### **Contoh Target Keuntungan Bersih Hipotetis (Q4 2026)**

Misalnya, Toko Niscare menetapkan target 